# NpuKit — transformer glue on PYNQ-Z2

Loads `npukit.bit` (VERSION `0x300`) and runs visible cases:

1. **Offline reference** — residual / GELU / RMSNorm / Softmax / attn smoke (CPU)
2. **Board glue** — same ops on FPGA with full vector dumps
3. **GEMM tile** — identity×ramp still works on the same bitstream

Docs: `docs/transformer_split.md`, `docs/transformer_glue.md`.

Keep `npukit.bit`, `npukit.hwh`, `npukit_transformer.py`, and `npukit_matmul.py` beside this notebook.
After **Run All**, save the notebook so the dumps stay in the file.

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_transformer as nt

importlib.reload(nt)
print("npukit_transformer loaded; MAX_LEN=", nt.MAX_LEN)
print("BIT=", BIT)

npukit_transformer loaded; MAX_LEN= 16
BIT= /home/xilinx/jupyter_notebooks/npukit.bit


## Offline reference (CPU only)

No bitstream required for this cell.

In [2]:
rc = nt.run_ref_suite()
assert rc == 0, "reference suite failed"
print("ref suite return code:", rc)

=== reference glue (offline) ===
residual OK [1. 0. 1. 0.]
gelu    OK [ 0.32177734 -0.10424805  0.72729492 -0.32373047]
rmsnorm OK [ 0.52954102 -0.26489258  1.05932617 -1.58911133]
softmax OK [0.85865784 0.08561707 0.0428009  0.01289368] sum 0.999969482421875
attn ref OK shape=(8, 8) meanQ12=-12.4
ALL REF PASS
ref suite return code: 0


## Board: load bitstream + glue + GEMM

Prints X/Y/GAMMA, OUT_npu, OUT_ref, PASS/FAIL for each op, then a GEMM tile.

In [3]:
rc = nt.run_board(BIT)
assert rc == 0, "board suite failed"
print("board suite return code:", rc)

Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID OK version=0x00000300 features=0x00000003

=== residual ===
X int32: [2048, -1024, 4096, -6144]
X float(Q12): [0.5, -0.25, 1.0, -1.5]
Y int32: [2048, 1024, 0, 6144]
Y float(Q12): [0.5, 0.25, 0.0, 1.5]
OUT_npu int32: [4096, 0, 4096, 0]
OUT_npu float(Q12): [1.0, 0.0, 1.0, 0.0]
OUT_ref int32: [4096, 0, 4096, 0]
OUT_ref float(Q12): [1.0, 0.0, 1.0, 0.0]
residual: PASS  max|err|=0  tol=0

=== gelu ===
X int32: [2048, -1024, 4096, -6144]
X float(Q12): [0.5, -0.25, 1.0, -1.5]
OUT_npu int32: [1318, -427, 2979, -1326]
OUT_npu float(Q12): [0.321777, -0.104248, 0.727295, -0.32373]
OUT_ref int32: [1318, -427, 2979, -1326]
OUT_ref float(Q12): [0.321777, -0.104248, 0.727295, -0.32373]
gelu: PASS  max|err|=0  tol=64

=== rmsnorm ===
X int32: [2048, -1024, 4096, -6144]
X float(Q12): [0.5, -0.25, 1.0, -1.5]
GAMMA int32: [4096, 4096, 4096, 4096]
GAMMA float(Q12): [1.0, 1.0, 1.0, 1.0]
OUT_npu int32: [2169, -1085, 4339, -6509]
OUT_npu f